# 06 - Weather prep (departure-anchored, no row-level merge)

**Not a row-level merge onto the 20M+ OTP rows** -- The dataset I use for modeling will reduce 20m+ OTP rows to one row per train run, and will only need to contain info about weather as of/before scheduled departure. This script loads Charlotte's raw hourly weather table (NOAA) and precomputes trailing-window aggregates over the ~98K-row hourly table. 
`07` then does a single `merge_asof` of each run's scheduled-departure timestamp against this prepped table.

Windows: `asof` (nearest pre-departure reading), `3h`/`6h`/`24h` trailing aggregates (`.rolling(window, closed="left")`, excluding the matched reading's own value). One dominant aggregation per variable -- mean for temp/dewpoint, max for wind/gust/ice, sum for precip (accumulation), min for vis/ceiling (worst conditions in the window). Pressure/daily-SOD fields stay `asof`-only. Boolean "did X happen in window" flags (`any_precip`/`any_gust`/`any_ice`) at `6h`/`24h` only.

In [1]:
import pandas as pd

BASEPATH = "../data"

wdf = pd.read_parquet("../weather/final_weather_ks.parquet")
wdf = wdf.sort_values("DATE").reset_index(drop = True)
print(f"Weather DF: {wdf.shape}, {wdf['DATE'].min()} to {wdf['DATE'].max()}")

Weather DF: (98281, 22), 2016-12-31 19:54:00-05:00 to 2025-08-26 23:54:00-04:00


In [2]:
wdf_idx = wdf.set_index("DATE")

WINDOW_SPECS = {
    "3h": ["temp_c", "wind_speed_ms", "gust_ms", "precip_mm", "vis_m"],
    "6h": ["temp_c", "wind_speed_ms", "gust_ms", "precip_mm", "vis_m", "ceiling_m"],
    "24h": ["temp_c", "dewpt_c", "wind_speed_ms", "gust_ms", "precip_mm", "vis_m", "ceiling_m", "ice_accretion_cm"],
}
AGG_FUNC = {
    "temp_c": "mean", "dewpt_c": "mean",
    "wind_speed_ms": "max", "gust_ms": "max",
    "precip_mm": "sum",
    "vis_m": "min", "ceiling_m": "min",
    "ice_accretion_cm": "max",
}

wdf_aug = wdf_idx.copy()
for window, cols in WINDOW_SPECS.items():
    for col in cols:
        agg = AGG_FUNC[col]
        wdf_aug[f"{col}_{window}"] = wdf_idx[col].rolling(window, closed = "left").agg(agg)

# "did X happen in the trailing window" boolean flags, 6h/24h only
wdf_idx["_has_precip"] = (wdf_idx["precip_mm"] > 0).astype(float)
wdf_idx["_has_gust"] = (wdf_idx["gust_ms"] > 0).astype(float)
wdf_idx["_has_ice"] = (wdf_idx["ice_accretion_cm"] > 0).astype(float)

for window in ["6h", "24h"]:
    for flag_col, out_name in [("_has_precip", "any_precip"), ("_has_gust", "any_gust"), ("_has_ice", "any_ice")]:
        wdf_aug[f"{out_name}_{window}"] = (
            wdf_idx[flag_col].rolling(window, closed = "left").max() > 0
        )

wdf_aug = wdf_aug.reset_index()
print(f"Weather feature table: {wdf_aug.shape}")

Weather feature table: (98281, 47)


In [ ]:
# carry the raw reading through as "_asof" (current condition at/before
# departure) for every column not already given a windowed name above
ASOF_COLS = [
    "temp_c", "dewpt_c", "wind_dir", "wind_speed_ms", "gust_ms", "slp_hpa",
    "pressure_tendency", "pressure_3hr_change_hpa", "ceiling_m", "vis_m",
    "cloud_cover_1", "cloud_base_m_1", "precip_mm", "precip_trace",
    "ice_accretion_cm", "wx_primary",
    "daily_snow_depth_cm", "daily_snow_24h_mm", "daily_peak1min_ms", "daily_mean_wind_ms",
]
wdf_aug = wdf_aug.rename(columns = {c: f"{c}_asof" for c in ASOF_COLS})

# dtype match for the merge_asof done in 07
wdf_aug["DATE"] = wdf_aug["DATE"].astype("datetime64[ns, America/New_York]")

weather_cols = [c for c in wdf_aug.columns if c != "DATE"]
print(f"{len(weather_cols)} weather feature columns built")

46 weather feature columns built


`wx_primary_asof` condition codes, per the abbreviation key in Charlotte's `weather/3_weather_explore_ks.r`:

| Code | Meaning | Code | Meaning |
|---|---|---|---|
| HR | heavy rain | TSLR | thunderstorm, slight rain |
| MR | moderate rain | TSMR | thunderstorm, moderate rain |
| SR | slight rain | TSHY | thunderstorm, heavy (rain) |
| HDZ | heavy drizzle | TSHL | thunderstorm, hail |
| DZ | slight drizzle | FG | fog |
| HS | heavy snow | DFG | dense fog |
| MS | moderate snow | MI | mist |
| SS | slight snow | HZ | haze |
| FR | freezing rain | SM | smoke |
| RS | mixed rain/snow | SQ | squalls |
| IP | ice pellets | LTG | lightning |
| SP | snow pellets | DU | slight dust |
| IC | ice crystals | SDU | severe dust |
| HL | hail | PU | precip, unclassified |
| TS | thunderstorm only | | |

`NaN` = no significant weather reported.

In [4]:
wdf_aug.to_parquet(f"{BASEPATH}/6_weather_prepped.parquet")
print(f"Saved {len(wdf_aug):,} rows to 6_weather_prepped.parquet")

Saved 98,281 rows to 6_weather_prepped.parquet
